# Sentiment Classification


1. **Import test and train data**
2. **Import the labels ( train and test)**
3. **Get the word index and then Create a key-value pair for word and word_id (20 points)**
4. **Build a Sequential Model using Keras for the Sentiment Classification task (20 points)**
5. **Report the Accuracy of the model (10 points)**
6.**Retrieve the output of each layer in Keras for a given single test sample from the trained model you built (10 points)**

## Loading the dataset

In [1]:
import numpy as np
from keras.datasets import imdb

np_load_old = np.load
np.load = lambda *a,**k: np_load_old(*a, allow_pickle=True, **k)

vocab_size = 10000 #vocab size

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size) # vocab_size is no.of words to consider from the dataset, ordering based on frequency.
np.load = np_load_old

Using TensorFlow backend.


17473536/17464789 [==============================] - 1s 0us/step


In [2]:
x_train.shape

(25000,)

In [3]:
x_test.shape

(25000,)

In [4]:

word_index = imdb.get_word_index()
word_and_word_index = dict(
[(value, key) for (key, value) in word_index.items()])

decoded_review = ' '.join(
[word_and_word_index.get(i - 3, '?') for i in x_train[10]])
print('---Words---')
print(decoded_review)
print('---Label---')
print(y_train[2])

1654784/1641221 [==============================] - 0s 0us/step
---Words---
? french horror cinema has seen something of a revival over the last couple of years with great films such as inside and ? romance ? on to the scene ? ? the revival just slightly but stands head and shoulders over most modern horror titles and is surely one of the best french horror films ever made ? was obviously shot on a low budget but this is made up for in far more ways than one by the originality of the film and this in turn is ? by the excellent writing and acting that ensure the film is a winner the plot focuses on two main ideas prison and black magic the central character is a man named ? sent to prison for fraud he is put in a cell with three others the quietly insane ? body building ? marcus and his retarded boyfriend daisy after a short while in the cell together they stumble upon a hiding place in the wall that contains an old ? after ? part of it they soon realise its magical powers and realise th

In [5]:
len(word_and_word_index)

88584

In [0]:
from keras.preprocessing.sequence import pad_sequences
vocab_size = 10000 #vocab size
maxlen = 300  #number of word used from each review

## Train test split

In [0]:
#load dataset as a list of ints
# (x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)
#make all sequences of the same length
x_train = pad_sequences(x_train, maxlen=maxlen)
x_test =  pad_sequences(x_test, maxlen=maxlen)

In [8]:
print(x_train)

[[   0    0    0 ...   19  178   32]
 [   0    0    0 ...   16  145   95]
 [   0    0    0 ...    7  129  113]
 ...
 [   0    0    0 ...    4 3586    2]
 [   0    0    0 ...   12    9   23]
 [   0    0    0 ...  204  131    9]]


In [9]:
print(x_test)

[[   0    0    0 ...   14    6  717]
 [   0    0    0 ...  125    4 3077]
 [1239 5189  137 ...    9   57  975]
 ...
 [   0    0    0 ...   21  846 5518]
 [   0    0    0 ... 2302    7  470]
 [   0    0    0 ...   34 2005 2643]]


## Build Keras Embedding Layer Model
We can think of the Embedding layer as a dicionary that maps a index assigned to a word to a word vector. This layer is very flexible and can be used in a few ways:

* The embedding layer can be used at the start of a larger deep learning model. 
* Also we could load pre-train word embeddings into the embedding layer when we create our model.
* Use the embedding layer to train our own word2vec models.

The keras embedding layer doesn't require us to onehot encode our words, instead we have to give each word a unqiue intger number as an id. For the imdb dataset we've loaded this has already been done, but if this wasn't the case we could use sklearn [LabelEncoder](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html).

In [0]:
from keras import Sequential
from keras.layers import Embedding, LSTM, Dense, Dropout
embedding_size=32

In [12]:

model=Sequential()
model.add(Embedding(vocab_size, embedding_size, input_length=maxlen))
model.add(LSTM(100))
model.add(Dense(1, activation='sigmoid'))


W0112 17:02:12.810534 140234290263936 module_wrapper.py:139] From /usr/local/lib/python2.7/dist-packages/keras/backend/tensorflow_backend.py:74: The name tf.get_default_graph is deprecated. Please use tf.compat.v1.get_default_graph instead.

W0112 17:02:12.819551 140234290263936 module_wrapper.py:139] From /usr/local/lib/python2.7/dist-packages/keras/backend/tensorflow_backend.py:517: The name tf.placeholder is deprecated. Please use tf.compat.v1.placeholder instead.

W0112 17:02:12.824657 140234290263936 module_wrapper.py:139] From /usr/local/lib/python2.7/dist-packages/keras/backend/tensorflow_backend.py:4138: The name tf.random_uniform is deprecated. Please use tf.random.uniform instead.



In [13]:
model.compile(loss='binary_crossentropy', 
             optimizer='adam', 
             metrics=['accuracy'])
print(model.summary())

W0112 17:02:28.126127 140234290263936 module_wrapper.py:139] From /usr/local/lib/python2.7/dist-packages/keras/optimizers.py:790: The name tf.train.Optimizer is deprecated. Please use tf.compat.v1.train.Optimizer instead.

W0112 17:02:28.154891 140234290263936 module_wrapper.py:139] From /usr/local/lib/python2.7/dist-packages/keras/backend/tensorflow_backend.py:3376: The name tf.log is deprecated. Please use tf.math.log instead.

W0112 17:02:28.159576 140234290263936 deprecation.py:323] From /usr/local/lib/python2.7/dist-packages/tensorflow_core/python/ops/nn_impl.py:183: where (from tensorflow.python.ops.array_ops) is deprecated and will be removed in a future version.
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where


_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_1 (Embedding)      (None, 300, 32)           320000    
_________________________________________________________________
lstm_1 (LSTM)                (None, 100)               53200     
_________________________________________________________________
dense_1 (Dense)              (None, 1)                 101       
Total params: 373,301
Trainable params: 373,301
Non-trainable params: 0
_________________________________________________________________
None


In [14]:
batch_size = 64
num_epochs = 10
x_test, y_test = x_train[:batch_size], y_train[:batch_size]
x_train2, y_train2 = x_train[batch_size:], y_train[batch_size:]
model.fit(x_train2, y_train2, validation_data=(x_test, y_test), batch_size=batch_size, epochs=num_epochs)


W0112 17:02:39.469002 140234290263936 module_wrapper.py:139] From /usr/local/lib/python2.7/dist-packages/keras/backend/tensorflow_backend.py:986: The name tf.assign_add is deprecated. Please use tf.compat.v1.assign_add instead.

W0112 17:02:39.551176 140234290263936 module_wrapper.py:139] From /usr/local/lib/python2.7/dist-packages/keras/backend/tensorflow_backend.py:973: The name tf.assign is deprecated. Please use tf.compat.v1.assign instead.

W0112 17:02:39.601488 140234290263936 module_wrapper.py:139] From /usr/local/lib/python2.7/dist-packages/keras/backend/tensorflow_backend.py:2741: The name tf.Session is deprecated. Please use tf.compat.v1.Session instead.

W0112 17:02:39.606528 140234290263936 module_wrapper.py:139] From /usr/local/lib/python2.7/dist-packages/keras/backend/tensorflow_backend.py:174: The name tf.get_default_session is deprecated. Please use tf.compat.v1.get_default_session instead.

W0112 17:02:39.607353 140234290263936 module_wrapper.py:139] From /usr/local/li

Train on 24936 samples, validate on 64 samples
Epoch 1/10


W0112 17:02:41.017057 140234290263936 module_wrapper.py:139] From /usr/local/lib/python2.7/dist-packages/keras/backend/tensorflow_backend.py:190: The name tf.global_variables is deprecated. Please use tf.compat.v1.global_variables instead.

W0112 17:02:41.018249 140234290263936 module_wrapper.py:139] From /usr/local/lib/python2.7/dist-packages/keras/backend/tensorflow_backend.py:199: The name tf.is_variable_initialized is deprecated. Please use tf.compat.v1.is_variable_initialized instead.

W0112 17:02:41.172281 140234290263936 module_wrapper.py:139] From /usr/local/lib/python2.7/dist-packages/keras/backend/tensorflow_backend.py:206: The name tf.variables_initializer is deprecated. Please use tf.compat.v1.variables_initializer instead.



24936/24936 [==============================] - 169s 7ms/step - loss: 0.5048 - acc: 0.7435 - val_loss: 0.2220 - val_acc: 0.9375
Epoch 2/10
24936/24936 [==============================] - 157s 6ms/step - loss: 0.2766 - acc: 0.8897 - val_loss: 0.2203 - val_acc: 0.9531
Epoch 3/10
24936/24936 [==============================] - 157s 6ms/step - loss: 0.2053 - acc: 0.9225 - val_loss: 0.2632 - val_acc: 0.9219
Epoch 4/10
24936/24936 [==============================] - 157s 6ms/step - loss: 0.1590 - acc: 0.9410 - val_loss: 0.2983 - val_acc: 0.8750
Epoch 5/10
24936/24936 [==============================] - 159s 6ms/step - loss: 0.1326 - acc: 0.9528 - val_loss: 0.3562 - val_acc: 0.8906
Epoch 6/10
24936/24936 [==============================] - 156s 6ms/step - loss: 0.1036 - acc: 0.9638 - val_loss: 0.4549 - val_acc: 0.9062
Epoch 7/10
24936/24936 [==============================] - 156s 6ms/step - loss: 0.0755 - acc: 0.9746 - val_loss: 0.4270 - val_acc: 0.9219
Epoch 8/10
24936/24936 [=====================

In [15]:
score, acc = model.evaluate(x_test, y_test,
                            batch_size=batch_size)
print('Test Score:', score)
print('Test Accuracy:', acc)

64/64 [==============================] - 0s 2ms/step
('Test Score:', 0.473749577999115)
('Test Accuracy:', 0.875)


In [0]:
import string
import nltk
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import gensim
from keras.initializers import Constant

In [17]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [18]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [0]:
review_lines = list()
lines = word_and_word_index.values()

In [0]:
def remove_special_characters(text, remove_digits=False):
    pattern = r'[^a-zA-z0-9\s]' if not remove_digits else r'[^a-zA-z\s]'
    text = re.sub(pattern, '', text)
    return text

In [0]:
for line in lines:
  tokens = word_tokenize(line)
  #convert to lowercase
  tokens = [w.lower() for w in tokens]
  #remove puchation and numbers from each word
  words = [remove_special_characters(w,remove_digits=True) for w in tokens]
  #filter out stopwords
  stop_words = set(stopwords.words('english'))
  words = [w for w in words if not w in stop_words]
  review_lines.append(words)

In [22]:
len(review_lines)

88584

In [0]:
#create and train word2vec model
model = gensim.models.Word2Vec(sentences = review_lines, size=embedding_size, window=5, min_count=1, workers=4)

In [24]:
words = list(model.wv.vocab)
print("vocabilary size : %d"%len(words))

vocabilary size : 73723


In [25]:
model.wv.most_similar(positive = "terrible")

/usr/local/lib/python2.7/dist-packages/gensim/matutils.py:737: FutureWarning: Conversion of the second argument of issubdtype from `int` to `np.signedinteger` is deprecated. In future, it will be treated as `np.int64 == np.dtype(int).type`.
  if np.issubdtype(vec.dtype, np.int):


[(u'convents', 0.6671053767204285),
 (u'shriekfest', 0.6665855050086975),
 (u'quarrels', 0.6511058807373047),
 (u'titus', 0.629612147808075),
 (u'rouncewell', 0.6137032508850098),
 (u'fool', 0.604658842086792),
 (u'racy', 0.6045066118240356),
 (u'dandies', 0.6043086051940918),
 (u'wowser', 0.5990557670593262),
 (u'normalos', 0.5986171960830688)]

In [26]:
#save model
filename = "imdb_embedding_word2vec.txt"
model.wv.save_word2vec_format(filename,binary=False)

/usr/local/lib/python2.7/dist-packages/smart_open/smart_open_lib.py:398: UserWarning: This function is deprecated, use smart_open.open instead. See the migration notes for details: https://github.com/RaRe-Technologies/smart_open/blob/master/README.rst#migrating-to-the-new-open-function
  'See the migration notes for details: %s' % _MIGRATION_NOTES_URL


In [0]:
#creating embedding matrix
import os
embedding_index = {}
f = open(os.path.join('','imdb_embedding_word2vec.txt'))
for line in f:
  values = line.split()
  word = values[0]
  coefs = np.asarray(values[1:])
  embedding_index[word]=coefs
f.close()

In [0]:
num_words = len(word_and_word_index) + 1
embedding_matrix = np.zeros((num_words, embedding_size))
for word, i in word_and_word_index.items():
  if i > num_words:
    continue
  embedding_vector = embedding_index.get(word)
  if embedding_vector is not None:
    embedding_matix[i]=embedding_vector

In [29]:
#Bulding model using Pre-trained Embedding
model=Sequential()
embedding_layer = Embedding(num_words, embedding_size, weights=[embedding_matrix], input_length=maxlen, trainable = False)
model.add(embedding_layer)
model.add(GRU(units=32,dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))
print(model.summary())

W0112 17:31:31.801651 140234290263936 module_wrapper.py:139] From /usr/local/lib/python2.7/dist-packages/keras/backend/tensorflow_backend.py:133: The name tf.placeholder_with_default is deprecated. Please use tf.compat.v1.placeholder_with_default instead.

W0112 17:31:31.808444 140234290263936 deprecation.py:506] From /usr/local/lib/python2.7/dist-packages/keras/backend/tensorflow_backend.py:3445: calling dropout (from tensorflow.python.ops.nn_ops) with keep_prob is deprecated and will be removed in a future version.
Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.


_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_2 (Embedding)      (None, 300, 32)           2834720   
_________________________________________________________________
gru_1 (GRU)                  (None, 32)                6240      
_________________________________________________________________
dense_2 (Dense)              (None, 1)                 33        
Total params: 2,840,993
Trainable params: 6,273
Non-trainable params: 2,834,720
_________________________________________________________________
None


In [30]:
model.compile(loss='binary_crossentropy', 
             optimizer='adam', 
             metrics=['accuracy'])

batch_size = 64
num_epochs = 10
x_test, y_test = x_train[:batch_size], y_train[:batch_size]
x_train2, y_train2 = x_train[batch_size:], y_train[batch_size:]
model.fit(x_train2, y_train2, validation_data=(x_test, y_test), batch_size=batch_size, epochs=num_epochs)

Train on 24936 samples, validate on 64 samples
Epoch 1/10
24936/24936 [==============================] - 143s 6ms/step - loss: 0.6933 - acc: 0.4970 - val_loss: 0.6910 - val_acc: 0.6094
Epoch 2/10
24936/24936 [==============================] - 145s 6ms/step - loss: 0.6932 - acc: 0.4976 - val_loss: 0.6925 - val_acc: 0.6094
Epoch 3/10
24936/24936 [==============================] - 143s 6ms/step - loss: 0.6932 - acc: 0.4995 - val_loss: 0.6948 - val_acc: 0.3906
Epoch 4/10
24936/24936 [==============================] - 144s 6ms/step - loss: 0.6932 - acc: 0.5038 - val_loss: 0.6905 - val_acc: 0.6094
Epoch 5/10
24936/24936 [==============================] - 148s 6ms/step - loss: 0.6932 - acc: 0.4958 - val_loss: 0.6924 - val_acc: 0.6094
Epoch 6/10
24936/24936 [==============================] - 149s 6ms/step - loss: 0.6932 - acc: 0.4991 - val_loss: 0.6914 - val_acc: 0.6094
Epoch 7/10
24936/24936 [==============================] - 148s 6ms/step - loss: 0.6932 - acc: 0.4963 - val_loss: 0.6924 - val

## Retrive the output of each layer in keras for a given single test sample from the trained model you built

In [31]:
from keras import backend as K

inp = model.input                                           # input placeholder
outputs = [layer.output for layer in model.layers]          # all layer outputs
functors = [K.function([inp, K.learning_phase()], [out]) for out in outputs]    # evaluation functions

# Testing
test = np.random.random(num_words)[np.newaxis,...]
layer_outs = [func([test, 1.]) for func in functors]
print layer_outs

[[array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]], dtype=float32)], [array([[ 0.0196509 ,  0.01361837,  0.00239031, -0.01708291,  0.01511581,
         0.00394731, -0.00299277, -0.00870145, -0.00734622, -0.01441351,
         0.00179495,  0.00779336,  0.00146253, -0.00165161,  0.00279457,
         0.0035268 ,  0.01842063,  0.00567366,  0.00258393, -0.00056466,
         0.00034318,  0.00045021, -0.00843083,  0.00422114,  0.00293113,
         0.00610629, -0.00055027, -0.00899252,  0.0159967 ,  0.00817755,
         0.00559119,  0.02437505]], dtype=float32)], [array([[0.50102985]], dtype=float32)]]
